<a href="https://colab.research.google.com/github/zohaib-mzg/Flyrank-ML-Internship/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 6: Validation and Research Claim Audit

This week I read the FlyRank research paper the way the live session walked through it, then turned that same reading on my own Week 5 model.

## 1. Two paper findings and my methodology questions

### Finding 1, What Predicts Health (page 27)

The paper reports a Random Forest with Average Position at 43 percent importance, Impressions at 32 percent, Scroll Depth at 15 percent, and CTR at 8 percent, predicting health score. To its credit, the paper already discloses the core issue itself, it states plainly that health score is partly constructed from some of these same inputs, so importance is descriptive rather than causal, and that high importance is expected rather than a sign of external causation. That disclosure is exactly the right instinct, and it's more caution than a lot of published work bothers with.

My methodology question builds on that disclosure rather than repeating it. If health score is a known weighted combination of position, impressions, CTR, and scroll depth, what does feature importance look like with those four inputs removed entirely, leaving only the features that aren't part of the label's own formula, content age, word count, days visible, AI sessions? Right now those four sit at 0 percent importance in the chart, which could mean they truly carry no signal, or it could just mean the four label-derived features are so dominant that nothing else gets a chance to show up. Those are different findings, and only a version of the model without the circular inputs can tell them apart.

### Finding 2, What Predicts Growth (page 29)

The paper reports a logistic regression reaching 71 percent holdout accuracy separating growing from declining pages, with content age as the strongest negative signal and days visible and recent impressions among the strongest positive signals.

My methodology question here is about the split, not the coefficients. The paper says holdout tested but doesn't say whether that holdout was grouped by client or domain, time based, or a plain random row level split. I'm asking this respectfully and concretely because I just found the answer matters a lot on my own data, see Section 2 below. If many pages in the sample come from a shared set of clients, and the split doesn't account for that, part of the 71 percent could reflect the model recognizing account level patterns rather than a genuine visibility signal. I'd want to know the grouping unit before treating that number as a stable estimate of how well this generalizes to a page from a brand new client.

## 2. My model under an honest split, before and after

In [1]:
import os, subprocess
import numpy as np
import pandas as pd

REPO_URL = "https://github.com/zohaib-mzg/Flyrank-ML-Internship"
REPO_DIR = "Flyrank-ML-Internship"

if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('../..')
elif not os.path.exists('data/raw/content_refresh_anonymized.csv'):
    if not os.path.isdir(REPO_DIR):
        subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)

df = pd.read_csv('data/raw/content_refresh_anonymized.csv')
df['is_declining'] = (df['trend_direction'] == 'down').astype(int)
print(f"clients: {df['client_id'].nunique()}, median pages per client: {df.groupby('client_id').size().median():.0f}")

clients: 32, median pages per client: 567


Same features and same model as Week 5, no changes there. The only thing changing is the split. Week 5 used a plain stratified random 75 25 split and named the client leakage risk without fixing it. Here I fix it, using `GroupShuffleSplit` so every page from a given client lands entirely in train or entirely in test, never both.

In [2]:
numeric_features = ['search_volume', 'competition', 'cpc', 'word_count', 'char_count',
    'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d',
    'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d',
    'days_with_impressions', 'days_with_sessions', 'content_age_days',
    'days_since_last_update', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct']

categorical_features = ['competition_level', 'content_type', 'main_intent', 'age_tier',
    'freshness_tier', 'word_count_tier', 'char_count_tier', 'impression_tier', 'position_tier']

feature_df = df[numeric_features + categorical_features].copy()
for col in numeric_features:
    feature_df[col] = feature_df[col].fillna(feature_df[col].median())

X = pd.get_dummies(feature_df, columns=categorical_features)
y = df['is_declining']

peer_tier_avg_ctr = df.groupby('position_tier')['ctr'].transform('mean')
qualifies = (df['impressions_90d'] >= 500) & (df['avg_position'] > 0) & (df['avg_position'] <= 20) & (df['ctr'] < 0.5)
df['baseline_score'] = np.where(qualifies, (peer_tier_avg_ctr - df['ctr']) * df['impressions_90d'], -1)

In [3]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier

def precision_at_k(scores, labels, k=50):
    order = np.argsort(-np.asarray(scores))[:k]
    return np.asarray(labels)[order].mean()

gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=df['client_id']))
idx_test = df.index[test_idx]

train_clients = set(df.loc[df.index[train_idx], 'client_id'])
test_clients = set(df.loc[idx_test, 'client_id'])
print(f"train clients: {len(train_clients)}, test clients: {len(test_clients)}")
print(f"client overlap between train and test: {train_clients & test_clients}")

rf = RandomForestClassifier(n_estimators=300, max_depth=8, random_state=42, n_jobs=-1)
rf.fit(X.iloc[train_idx], y.iloc[train_idx])
proba_single = rf.predict_proba(X.iloc[test_idx])[:, 1]

p50_rf_single = precision_at_k(proba_single, y.iloc[test_idx].values, 50)
p50_baseline_single = precision_at_k(df.loc[idx_test, 'baseline_score'].values, y.iloc[test_idx].values, 50)
print(f"single grouped split, seed 42, Precision@50: baseline {p50_baseline_single:.3f}, model {p50_rf_single:.3f}")

train clients: 24, test clients: 8
client overlap between train and test: set()
single grouped split, seed 42, Precision@50: baseline 0.500, model 0.480


One grouped split confirms zero client overlap, the leak is genuinely closed. But a single split like this depends heavily on which 8 clients happen to land in the test set out of only 32 total. Before reporting one number as the honest result, I checked how much it moves across different client splits.

In [5]:
rf_scores, baseline_scores = [], []
for seed in range(20):
    gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=seed)
    tr_idx, te_idx = next(gss.split(X, y, groups=df['client_id']))
    rf_i = RandomForestClassifier(n_estimators=300, max_depth=8, random_state=42, n_jobs=-1)
    rf_i.fit(X.iloc[tr_idx], y.iloc[tr_idx])
    proba_i = rf_i.predict_proba(X.iloc[te_idx])[:, 1]
    idx_te = df.index[te_idx]
    rf_scores.append(precision_at_k(proba_i, y.iloc[te_idx].values, 50))
    baseline_scores.append(precision_at_k(df.loc[idx_te, 'baseline_score'].values, y.iloc[te_idx].values, 50))

rf_scores = np.array(rf_scores)
baseline_scores = np.array(baseline_scores)

before_after = pd.DataFrame({
    'split': ['Week 5, random split (single)', 'Week 6, grouped split (single seed)', 'Week 6, grouped split (20 seeds, mean)'],
    'baseline_precision_at_50': [0.52, round(p50_baseline_single, 3), round(baseline_scores.mean(), 3)],
    'model_precision_at_50': [0.88, round(p50_rf_single, 3), round(rf_scores.mean(), 3)],
})
before_after

,split,baseline_precision_at_50,model_precision_at_50
0,"Week 5, random split (single)",0.520,0.880
1,"Week 6, grouped split (single seed)",0.500,0.480
2,"Week 6, grouped split (20 seeds, mean)",0.531,0.761


In [6]:
print(f"model precision@50 across 20 grouped splits: mean {rf_scores.mean():.3f}, std {rf_scores.std():.3f}, min {rf_scores.min():.3f}, max {rf_scores.max():.3f}")
print(f"baseline precision@50 across 20 grouped splits: mean {baseline_scores.mean():.3f}, std {baseline_scores.std():.3f}")

model precision@50 across 20 grouped splits: mean 0.761, std 0.147, min 0.400, max 1.000
baseline precision@50 across 20 grouped splits: mean 0.531, std 0.141


Here's the honest before and after. Week 5's random split reported 0.88. The single grouped split above landed at 0.48, which on its own would have looked like a much worse story than it actually is. Averaged across 20 different grouped splits, the model settles at a mean of 0.761 with a standard deviation of 0.147, ranging as low as 0.40 and as high as 1.00 depending on which clients ended up in the test set. The baseline moves too, averaging 0.531 under the same grouped splits, close to what it scored on the plain random split.

Two real conclusions come out of this, not one. First, the client leakage risk I flagged in Week 5 was real, 0.88 on a random split was an optimistic number, and the honest grouped estimate is meaningfully lower. Second, with only 32 clients total, any single grouped split is itself unstable, so the right way to report this isn't the model's grouped score, it's the model's grouped score averaged across many client splits, with the spread reported alongside it. The model still beats the baseline on average, 0.761 against 0.531, but that gap is smaller and comes with real uncertainty that the Week 5 number hid completely.

## 3. Leakage audit

In [7]:
leakage_check = df[numeric_features].corrwith(df['is_declining']).sort_values(key=abs, ascending=False)
leakage_check

,0
days_with_impressions,0.190055
content_age_days,-0.163882
word_count,0.090157
days_since_last_update,0.081383
char_count,0.072188
ctr,-0.061911
clicks_90d,-0.039680
engaged_sessions_90d,-0.035402
avg_position,-0.029035
days_with_sessions,-0.025055


This repeats the same style of check I ran in Week 3 on the warehouse data, correlation between each candidate feature and the label, looking for anything close to 1 that would mean a feature is really just the label wearing a different name. The strongest correlation here is `days_with_impressions` at 0.19, followed by `content_age_days` at negative 0.16. Both are modest, in the same range as the 0.177 I found for `log_impressions_90d` back in Week 3, not the near perfect correlation a real leak would produce.

The feature list itself is unchanged from Week 5, `trend_direction` and `trend_pct` excluded as the label's own source, and the six last30 and prev30 columns excluded because they're literally the two windows `trend_direction` is computed from. Nothing in this correlation table changes that decision, if anything it confirms it, since none of the remaining features come anywhere close to acting like a hidden copy of the label.

## 4. Claim rewrite

Two claims from my Week 5 notebook go further than the evidence now supports, rewritten here with safer language.

Original, Week 5: Random Forest reaches 0.90, clear the baseline by a wide margin.
Rewritten: on a client grouped split, averaged across 20 different client splits, the model's measured Precision@50 is 0.761 against a baseline of 0.531, a real but smaller and less certain advantage than the single random split suggested, with a standard deviation of 0.15 across splits.

Original, Week 5: the model correctly learning a real pattern.
Rewritten: the model's errors are associated with a pattern in the training data, staleness combined with volume and position, that does not hold for every page it matches. This is an observed association in this sample, not a claim about why any individual page changed.

Both rewrites move from a confident, general statement to an observed, sample specific, uncertainty aware one, the same standard I asked of the paper's own findings in Section 1.

## 5. Self-check

Two paper findings named with a real page reference each, and both methodology questions are built on top of what the paper already discloses rather than treating an already disclosed limitation as a gotcha.

My own model re-run under a client grouped split, with a genuine before and after, 0.88 random split versus 0.761 mean grouped split, and the single split versus 20 split averaging shown as its own separate lesson, not folded quietly into one number.

Leakage audit re-checked with actual correlations against the label, not just a repeated assertion that the Week 5 exclusions were enough.

Two claims rewritten with observed, measured, and directional language, matching the same standard applied to the paper.

Every number in this notebook is real, computed from the same starter CSV used since Week 2, nothing here is estimated or asserted without a cell producing it above.